
# 🤖 MGMT 467 - Unit 2 Lab 2: Prompt Studio for AI-Assisted SQL + ML

**Date:** 2025-10-16  
**Objective:** Build and refine a complete ML pipeline for churn prediction using BigQuery — but with **Gemini-style prompts** guiding SQL generation.

You'll learn to:
- Frame SQL goals as clear prompts
- Generate, test, and debug queries with an AI assistant
- Reflect on each modeling step and your prompt design



## Task 0: Connect to BigQuery

**🎯 Goal:** Verify BigQuery access from Colab.  
**📌 Requirements:** Use `%%bigquery`, get current date and user session.

---

### 🧠 Prompt Template  
> Write a SQL query that returns CURRENT_DATE() and SESSION_USER(). I will run it with %%bigquery in Colab.

---

### 👩‍🏫 Example Prompt  
> Write a SQL query using BigQuery syntax that returns today’s date and the current session user.

---

### ✅ Expected SQL Output
```sql
SELECT CURRENT_DATE() AS today, SESSION_USER() AS user;
```

---

### 🔍 Checkpoint  
Query should return a single row with today's date and your user.


In [1]:
from google.colab import auth
from google.cloud import bigquery

# Authenticate with Google
auth.authenticate_user()

# Set your BigQuery project ID here
project_id = 'mgmt-467-471613' # Replace with your actual project ID

# Construct a BigQuery client object
client = bigquery.Client(project=project_id)

print(f"BigQuery client created with project: {client.project}")

BigQuery client created with project: mgmt-467-471613


In [2]:
%%bigquery --project mgmt-467-471613
SELECT CURRENT_DATE() AS today, SESSION_USER() AS user;

Query is running:   0%|          |

Downloading:   0%|          |

,today,user
0,2025-10-25,rileighdethy@gmail.com


You can set your BigQuery project ID in Colab by authenticating with Google and then specifying the project.


## Task 1: Prepare ML Table

**🎯 Goal:** Create a clean features table for modeling churn.  
**📌 Requirements:** Use cleaned_features as source, select relevant columns, filter rows with churn_label IS NOT NULL.

---

### 🧠 Prompt Template  
> Write a query that creates a new table with columns: [region, plan_tier, age_band, ...] and churn_label from [source_table]. Filter to rows where churn_label IS NOT NULL.

---

### 👩‍🏫 Example Prompt  
> Create a BigQuery table named churn_features from cleaned_features with selected features and where churn_label IS NOT NULL.

---

### ✅ Expected SQL Output
```sql
CREATE OR REPLACE TABLE `your_dataset.churn_features` AS
SELECT region, plan_tier, age_band, avg_rating, total_minutes, churn_label
FROM `your_dataset.cleaned_features`
WHERE churn_label IS NOT NULL;
```

---

### 🔍 Checkpoint  
Table should appear in BigQuery and contain non-null labels.


In [3]:
%%bigquery --project mgmt-467-471613
SELECT *
FROM `mgmt-467-471613.netflix.users`
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,email,first_name,last_name,age,gender,country,state_province,city,subscription_plan,subscription_start_date,is_active,monthly_spend,primary_device,household_size,created_at
0,user_00342,christophervincent@example.com,Brittany,Ramirez,48.0,Female,Canada,Alberta,East Mark,Premium,2025-04-05,True,46.97,Desktop,NaN,2025-04-30 06:57:21.478166+00:00
1,user_00784,alexander25@example.org,Stacey,Cortez,25.0,Female,Canada,Alberta,Guzmanburgh,Basic,2024-01-10,True,16.38,Desktop,7.0,2024-12-21 04:06:48.998214+00:00
2,user_00988,sarahrollins@example.com,Evelyn,Hayes,33.0,Female,Canada,Alberta,East Elizabeth,Standard,2024-01-20,True,4.29,Desktop,1.0,2023-03-01 00:59:30.491893+00:00
3,user_01621,jeffreyfinley@example.org,Patrick,Hayes,55.0,Male,Canada,Alberta,West Christian,Basic,2024-02-21,True,11.17,Desktop,5.0,2025-01-24 04:58:30.868652+00:00
4,user_01821,grimeshenry@example.net,David,Trevino,31.0,Female,Canada,Alberta,South Angela,Premium,2024-11-14,True,8.08,Desktop,2.0,2023-11-13 13:44:03.953962+00:00
5,user_01956,michaelwood@example.org,William,Rush,39.0,Male,Canada,Alberta,New April,Standard,2022-09-01,True,29.72,Desktop,3.0,2023-07-15 11:38:13.512970+00:00
6,user_02082,schwartzmichael@example.com,Christopher,Perry,NaN,Male,Canada,Alberta,New Johnbury,Standard,2023-05-03,True,1.83,Desktop,5.0,2023-08-12 09:46:01.840363+00:00
7,user_02266,charleschambers@example.org,Charles,Huang,36.0,Female,Canada,Alberta,New Patriciatown,Premium+,2023-10-16,True,NaN,Desktop,3.0,2023-07-09 00:49:41.724081+00:00
8,user_02569,yalexander@example.com,Shelley,Gray,27.0,None,Canada,Alberta,Port Brittanymouth,Standard,2023-11-25,True,18.26,Desktop,NaN,2023-04-04 19:13:57.702305+00:00
9,user_02677,gwagner@example.com,Kyle,Macdonald,15.0,Male,Canada,Alberta,Lake Tommyton,Premium+,2025-07-12,False,5.81,Desktop,3.0,2023-08-13 14:47:52.253552+00:00


In [12]:
%%bigquery --project mgmt-467-471613
CREATE OR REPLACE TABLE `netflix.churn_features` AS
SELECT
    u.user_id,
    u.state_province,
    u.subscription_plan,
    u.age,
    AVG(r.rating) AS avg_rating,
    SUM(w.watch_duration_minutes) AS total_minutes,
    u.is_active AS churn_label
FROM
    `netflix.users` AS u
LEFT JOIN
    `netflix.reviews` AS r ON u.user_id = r.user_id
LEFT JOIN
    `netflix.watch_history` AS w ON u.user_id = w.user_id
WHERE u.is_active IS NOT NULL -- Assuming is_active is the churn label here
GROUP BY
    u.user_id, u.state_province, u.subscription_plan, u.age, u.is_active;

Query is running:   0%|          |

""


In [13]:
%%bigquery --project mgmt-467-471613
SELECT *
FROM `netflix.churn_features`
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,state_province,subscription_plan,age,avg_rating,total_minutes,churn_label
0,user_01012,Alberta,Basic,NaN,NaN,3585.6,True
1,user_04708,Alberta,Premium,NaN,4.333333,43318.8,True
2,user_03181,Alberta,Basic,NaN,4.000000,18929.7,True
3,user_07071,Alberta,Standard,NaN,4.000000,31762.8,True
4,user_08635,Alberta,Basic,NaN,NaN,5371.2,True
5,user_03179,Alberta,Premium,NaN,2.000000,21821.4,True
6,user_06657,Alberta,Standard,NaN,3.000000,10986.3,True
7,user_04243,Alberta,Standard,NaN,NaN,5535.9,False
8,user_03384,Alberta,Premium+,NaN,4.000000,11672.1,True
9,user_08374,Alberta,Premium,NaN,3.000000,10824.3,False



## Task 2: Train Logistic Regression Model

**🎯 Goal:** Train a basic BQML logistic regression model.  
**📌 Requirements:** Use churn_features table, predict churn_label from features.

---

### 🧠 Prompt Template  
> Write a CREATE MODEL SQL for logistic regression using churn_label as label and [features] as inputs.

---

### 👩‍🏫 Example Prompt  
> Train a logistic regression model to predict churn_label using region, plan_tier, total_minutes, avg_rating.

---

### ✅ Expected SQL Output
```sql
CREATE OR REPLACE MODEL `your_dataset.churn_model`
OPTIONS(model_type='logistic_reg') AS
SELECT region, plan_tier, total_minutes, avg_rating, churn_label
FROM `your_dataset.churn_features`;
```

---

### 🔍 Checkpoint  
Model appears in BigQuery under Models. Training completes.


In [16]:
%%bigquery --project mgmt-467-471613
CREATE OR REPLACE MODEL `netflix.churn_model`
OPTIONS(model_type='logistic_reg', input_label_cols=['churn_label']) AS
SELECT
    state_province,
    subscription_plan,
    age,
    avg_rating,
    total_minutes,
    churn_label
FROM
    `netflix.churn_features`;

Query is running:   0%|          |

""



## Task 3: Evaluate Model

**🎯 Goal:** Evaluate the logistic regression model.  
**📌 Requirements:** Use ML.EVALUATE.

---

### 🧠 Prompt Template  
> Write a query to evaluate my logistic regression model using ML.EVALUATE.

---

### 👩‍🏫 Example Prompt  
> Evaluate the churn_model using ML.EVALUATE to get accuracy, precision, recall.

---

### ✅ Expected SQL Output
```sql
SELECT * FROM ML.EVALUATE(MODEL `your_dataset.churn_model`);
```

---

### 🔍 Checkpoint  
View performance metrics: accuracy, log_loss, precision, recall.


In [17]:
%%bigquery --project mgmt-467-471613
SELECT * FROM ML.EVALUATE(MODEL `netflix.churn_model`);

Query is running:   0%|          |

Downloading:   0%|          |

,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.860465,1.0,0.860465,0.925,0.404703,0.501328



## Task 4: Predict Churn

**🎯 Goal:** Use ML.PREDICT to generate churn predictions.  
**📌 Requirements:** Apply model to same input table.

---

### 🧠 Prompt Template  
> Generate SQL to use ML.PREDICT on churn_model and return predictions by user_id.

---

### 👩‍🏫 Example Prompt  
> Predict churn using churn_model. Include user_id, predicted_churn_label, and prediction probability.

---

### ✅ Expected SQL Output
```sql
SELECT user_id, predicted_churn_label, predicted_churn_label_probs
FROM ML.PREDICT(MODEL `your_dataset.churn_model`,
      (SELECT * FROM `your_dataset.churn_features`));
```

---

### 🔍 Checkpoint  
Inspect top churn risk users. Validate probabilities.


In [20]:
%%bigquery --project mgmt-467-471613
SELECT
    user_id,
    predicted_churn_label,
    predicted_churn_label_probs
FROM
    ML.PREDICT(MODEL `netflix.churn_model`,
      (SELECT * FROM `netflix.churn_features`))
ORDER BY
    predicted_churn_label_probs[OFFSET(0)].prob DESC
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,predicted_churn_label,predicted_churn_label_probs
0,user_01334,True,"[{'label': True, 'prob': 0.8917124724502516}, ..."
1,user_00751,True,"[{'label': True, 'prob': 0.89136642186657}, {'..."
2,user_01023,True,"[{'label': True, 'prob': 0.8912924081062096}, ..."
3,user_09121,True,"[{'label': True, 'prob': 0.8910388861992077}, ..."
4,user_06868,True,"[{'label': True, 'prob': 0.8909223335065323}, ..."
5,user_03957,True,"[{'label': True, 'prob': 0.8907089729024139}, ..."
6,user_02031,True,"[{'label': True, 'prob': 0.8905242162647686}, ..."
7,user_02908,True,"[{'label': True, 'prob': 0.8900137982478581}, ..."
8,user_03494,True,"[{'label': True, 'prob': 0.8897282113048384}, ..."
9,user_01176,True,"[{'label': True, 'prob': 0.8896895143482886}, ..."
